In [2]:
import os
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import OxfordIIITPet
import pathlib
from torch.utils.tensorboard import SummaryWriter
import torchinfo
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
import seaborn as sns
from pathlib import Path
from datetime import datetime
import json
from scipy.ndimage import gaussian_filter
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [3]:
# 1. Setup Base Paths
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

data_dir = Path("data")
base_dir = Path("segmentation")
run_dir  = base_dir / f"run_{timestamp}"

models = ["unet_ignore_border", "unet_weighted_border", "unet_balanced"]
dirs = {}

# 2. Define Hierarchy
for model in models:
    model_dir = run_dir / model
    dirs[model] = {
        "root": model_dir,
        "logs": model_dir / "logs",
        "checkpoints": model_dir / "checkpoints",
        "configs": model_dir / "configs",
        "results": model_dir / "results",
    }

# 3. Create folders
data_dir.mkdir(parents=True, exist_ok=True)

for model_paths in dirs.values():
    for path in model_paths.values():
        path.mkdir(parents=True, exist_ok=True)

# Example Usage:
print(f"UNET Logs will be saved to: {dirs['unet_balanced']['logs']}")


UNET Logs will be saved to: segmentation/run_20260512_133157/unet_balanced/logs


In [4]:
def seed_everything(seed=42):
    # 1. Basic Python and OS Level
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    
    # 2. Torch CPU
    torch.manual_seed(seed)
    
    # 3. NVIDIA CUDA (Nvidia GPUs)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # for multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
    # 4. Apple MPS (M1/M2/M3 Macs)
    if hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
        # Note: torch.use_deterministic_algorithms(True) often fails on MPS
        # because Metal shaders don't support it for all ops yet.
    
    # 5. Global Determinism (Optional/Strict)
    # This will throw an error if an op doesn't have a deterministic impl.
    # torch.use_deterministic_algorithms(True, warn_only=True)


In [5]:
seed_everything(42)

/leonardo/home/userinternal/msafari1/my_ai_base/build/lib/python3.10/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [6]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    else:
        return torch.device("cpu")

In [7]:
device = get_device()
print(device)

cpu


In [8]:
class PetUNet_Encoder(nn.Module):
    def __init__(self, n_channels=3, base_channel_size=32):
        super().__init__()
        
        self.inc = nn.Sequential(
            nn.Conv2d(n_channels, base_channel_size, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size), nn.GELU(),
            nn.Conv2d(base_channel_size, base_channel_size, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size), nn.GELU()
        )
        
        self.down1 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(base_channel_size, base_channel_size * 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2), nn.GELU(),
            nn.Conv2d(base_channel_size * 2, base_channel_size * 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2), nn.GELU()
        )
        
        self.down2 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(base_channel_size * 2, base_channel_size * 4, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4), nn.GELU(),
            nn.Conv2d(base_channel_size * 4, base_channel_size * 4, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4), nn.GELU()
        )
        
        self.down3 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(base_channel_size * 4, base_channel_size * 8, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 8), nn.GELU(),
            nn.Conv2d(base_channel_size * 8, base_channel_size * 8, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 8), nn.GELU()
        )
        
        self.down4 = nn.Sequential(
            nn.MaxPool2d(2),
            nn.Conv2d(base_channel_size * 8, base_channel_size * 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 16), nn.GELU(),
            nn.Conv2d(base_channel_size * 16, base_channel_size * 16, 3, padding=1, bias=False),
            # Output of bottleneck is base_channel_size * 16
            nn.BatchNorm2d(base_channel_size * 16), nn.GELU()
        )

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        return x5, x4, x3, x2, x1
    


class PetUNet_Decoder(nn.Module):
    def __init__(self, base_channel_size=32):
        super().__init__()
        # Input: x5 (16C) + x4 (8C) = 24C
        self.up_conv1 = nn.Sequential(
            nn.Conv2d(base_channel_size * 16 + base_channel_size * 8, base_channel_size * 8, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 8),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 8, base_channel_size * 8, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 8),
            nn.GELU()
        )
        # Input: x (8C) + x3 (4C) = 12C
        self.up_conv2 = nn.Sequential(
            nn.Conv2d(base_channel_size * 8 + base_channel_size * 4, base_channel_size * 4, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 4, base_channel_size * 4, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 4),
            nn.GELU()
        )
        # Input: x (4C) + x2 (2C) = 6C
        self.up_conv3 = nn.Sequential(
            nn.Conv2d(base_channel_size * 4 + base_channel_size * 2, base_channel_size * 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU(),
            nn.Conv2d(base_channel_size * 2, base_channel_size * 2, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size * 2),
            nn.GELU()
        )
        # Input: x (2C) + x1 (1C) = 3C
        self.up_conv4 = nn.Sequential(
            nn.Conv2d(base_channel_size * 2 + base_channel_size, base_channel_size, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU(),
            nn.Conv2d(base_channel_size, base_channel_size, 3, padding=1, bias=False),
            nn.BatchNorm2d(base_channel_size),
            nn.GELU()
        )

    def forward(self, x5, x4, x3, x2, x1):
        x = F.interpolate(x5, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat([x, x4], dim=1)
        x = self.up_conv1(x)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat([x, x3], dim=1)
        x = self.up_conv2(x)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat([x, x2], dim=1)
        x = self.up_conv3(x)
        x = F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False)
        x = torch.cat([x, x1], dim=1)
        x = self.up_conv4(x)
        return x
    
class PetUNet(nn.Module):
    def __init__(self, n_channels=3, n_classes=3, base_channel_size=32):
        super().__init__()
        self.encoder = PetUNet_Encoder(n_channels, base_channel_size)
        self.decoder = PetUNet_Decoder(base_channel_size)
        self.outc = nn.Conv2d(base_channel_size, n_classes, kernel_size=1)

    def forward(self, x):
        x5, x4, x3, x2, x1 = self.encoder(x)
        mask_features = self.decoder(x5, x4, x3, x2, x1)
        return self.outc(mask_features) # logits
    

In [9]:
#class PetSegmentationDataset(Dataset):
#    def __init__(self, dataset, image_size=128):
#        self.dataset = dataset
#        self.image_size = image_size
#
#    def __len__(self): return len(self.dataset)
#
#    def __getitem__(self, idx):
#        image, mask = self.dataset[idx]
#        image = image.resize((self.image_size, self.image_size))
#        mask = mask.resize((self.image_size, self.image_size), resample=Image.NEAREST)
#        
#        image = np.array(image).astype(np.float32) / 255.0
#        image = image.transpose(2, 0, 1)
#        mask = np.array(mask).astype(np.int64) - 1 # 1,2,3 -> 0,1,2
#        
#        return torch.tensor(image), torch.tensor(mask)


In [10]:
def multi_class_dice(logits, target, weights=None, ignore_index=None, eps=1e-7):
    C = logits.shape[1]
    probs = F.softmax(logits, dim=1)

    # 1. Pixel-level masking (Handling ignore_index)
    mask = torch.ones_like(target, dtype=torch.bool)
    if ignore_index is not None:
        mask = target != ignore_index

    # One-hot encoding safely
    safe_target = target.clone()
    safe_target[~mask] = 0
    target_1hot = F.one_hot(safe_target.long(), C).permute(0, 3, 1, 2).float()

    # Apply mask to both probs and target
    mask = mask.unsqueeze(1)
    probs = probs * mask
    target_1hot = target_1hot * mask

    # 2. Dice calculation per class
    dims = (0, 2, 3)
    inter = (probs * target_1hot).sum(dims)
    denom = probs.sum(dims) + target_1hot.sum(dims)
    dice_per_class = (2 * inter + eps) / (denom + eps)

    # 3. Apply Weighting Logic
    if weights is not None:
        weights = weights.to(logits.device)
        # If we are ignoring a class via weights (e.g., weight=0), 
        # it behaves like your previous 'valid_classes' filter.
        return (dice_per_class * weights).sum() / (weights.sum() + eps)

    return dice_per_class.mean()

class CombinedLoss(nn.Module):
    def __init__(self, mode="ignore_border", ce_weight=0.5):
        super().__init__()
        self.mode = mode
        self.ce_weight = ce_weight

        if mode == "ignore_border":
            # Ignore border (index 2) in CE and Dice
            # Weights: Background=1, Pet=1, Border=0
            weights = torch.tensor([1.0, 1.0, 0.0])
            self.register_buffer('weights', weights)
            self.ce = nn.CrossEntropyLoss(ignore_index=2)

        elif mode == "weighted_border":
            # Include border but emphasize it (1, 1, 2)
            weights = torch.tensor([1.0, 1.0, 2.0])
            self.register_buffer('weights', weights)
            self.ce = nn.CrossEntropyLoss(weight=self.weights)

        elif mode == "balanced":
            weights = torch.tensor([1.0, 1.0, 1.0])
            self.register_buffer('weights', weights)
            self.ce = nn.CrossEntropyLoss()

    def forward(self, logits, target):
        ce_loss = self.ce(logits, target)

        # Pass the weights buffer to the dice function
        # We also pass ignore_index=2 if we are in ignore mode
        ignore_idx = 2 if self.mode == "ignore_border" else None
        
        dice_score = multi_class_dice(
            logits, 
            target, 
            weights=self.weights, 
            ignore_index=ignore_idx
        )
        
        dice_loss = 1.0 - dice_score
        loss = self.ce_weight * ce_loss + (1 - self.ce_weight) * dice_loss

        return loss, dice_score

In [11]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = np.inf
        self.best_dice = -np.inf
        self.early_stop = False
        self.best_state = None
        self.best_epoch = -1

    def __call__(self, model, epoch, val_loss, val_dice):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_dice = val_dice
            self.counter = 0

            self.best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

    def load_best_weights(self, model):
        if self.best_state is not None:
            model.load_state_dict(self.best_state)
            print(f"Restored best model weights (Val Loss: {self.best_loss:.4f})")


In [12]:
def train(train_loader, val_loader, test_loader, 
          model, criterion, 
          optimizer, scheduler, 
          max_epochs, early_stopping,
          writer, checkpoint_filename, result_filename): 
    
    device = next(model.parameters()).device
    criterion.to(device)
    print(f"Training on {device}")
    
    for epoch in range(max_epochs):
        # -------- TRAIN --------
        model.train()
        train_loss_sum = 0.0
        train_dice_sum  = 0.0
        train_num_samples = 0
        train_num_batches = len(train_loader)
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch:03d} [Train]")
        for batch_idx, (imgs, msks) in enumerate(train_bar):
            batch_size = imgs.size(0)
            train_num_samples += batch_size
            # Move to MPS/GPU
            imgs = imgs.to(device)
            msks = msks.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            logits = model(imgs)
            loss, dice = criterion(logits, msks)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            # -------- Scheduler step (batch-wise only for CAWR) --------
            if scheduler is not None and isinstance(
                scheduler, torch.optim.lr_scheduler.CosineAnnealingWarmRestarts):
                scheduler.step(epoch + batch_idx / train_num_batches)
            
            train_loss_sum += loss.item() * batch_size
            train_dice_sum += dice.item() * batch_size
            train_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "lr": f"{optimizer.param_groups[0]['lr']:.2e}"
            })


        train_loss = train_loss_sum / train_num_samples
        train_dice = train_dice_sum / train_num_samples


        # -------- VALIDATION --------
        model.eval()
        val_num_samples = 0
        val_loss_sum = 0.0
        val_dice_sum = 0.0
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch:03d} [Val]")
        with torch.no_grad():
            for imgs, msks in val_bar:
                batch_size = imgs.size(0)
                val_num_samples += batch_size
                imgs = imgs.to(device)
                msks = msks.to(device)
                logits = model(imgs)
                loss, dice = criterion(logits, msks)
                val_loss_sum += loss.item() * batch_size
                val_dice_sum += dice.item() * batch_size
        
        val_loss = val_loss_sum / val_num_samples
        val_dice = val_dice_sum / val_num_samples

        # -------------------- SCHEDULER --------------------
        if scheduler is not None:
            if isinstance(
                scheduler,torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            elif not isinstance(
                scheduler,torch.optim.lr_scheduler.CosineAnnealingWarmRestarts):
                scheduler.step()
        # -------------------- LOGGING --------------------
        writer.add_scalar("loss/train",  train_loss, epoch)
        writer.add_scalar("loss/val",    val_loss,   epoch)
        writer.add_scalar("dice/train",  train_dice, epoch)
        writer.add_scalar("dice/val",    val_dice,   epoch)
        writer.add_scalar("lr", optimizer.param_groups[0]["lr"], epoch)

        print(
            f"Epoch {epoch:03d} | "
            f"Train: Loss = {train_loss:.4f} Dice = {train_dice:.4f} | "
            f"Val: Loss = {val_loss:.4f} Dice = {val_dice:.4f}"
        )
        
        # -------------------- EARLY STOPPING --------------------

        if early_stopping(model, epoch, val_loss, val_dice):
            print("Stopping early")
            break
            
    # Before saving or testing, revert to the best discovered state
    early_stopping.load_best_weights(model)
    ## -------------------- SAVE --------------------
    #torch.save(
    #    model.state_dict(),
    #    os.path.join(checkpoint_path, f"{model.name}_{model.latent_dim}.pt")
    #)
    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_epoch": early_stopping.best_epoch,
    }, checkpoint_filename)
    # -------------------- TEST --------------------
    model.eval()
    test_loss_sum = 0.0
    test_dice_sum = 0.0
    test_num_samples = 0
    with torch.no_grad():
        for imgs, msks in test_loader:
            batch_size = imgs.size(0)
            test_num_samples += batch_size
            imgs = imgs.to(device)
            msks = msks.to(device)
            logits = model(imgs)
            loss, dice = criterion(logits, msks)
            test_loss_sum += loss.item() * batch_size
            test_dice_sum += dice.item() * batch_size
    
    test_loss = test_loss_sum / test_num_samples
    test_dice = test_dice_sum / test_num_samples
    print(f"Test:\tLoss = {test_loss:.4f}\tDice = {test_dice:.4f}")  
    # After the test results are calculated
    
    result = {
        "test": {"loss": test_loss, "dice": test_dice},
        "val":  {"loss": early_stopping.best_loss, 
                 "dice": early_stopping.best_dice,
                 "best_epoch": early_stopping.best_epoch,
                },
    }
    # Save results next to the checkpoint
    with open(result_filename, "w") as f:
        json.dump(result, f, indent=4)
    
    print(f"Results saved to {result_filename}")
    return result


In [13]:
#import albumentations as A
#from albumentations.pytorch import ToTensorV2

# --- 1. Dataset Class & Transforms ---
class AlbumentationsPetDataset(Dataset):
    def __init__(self, root, split="trainval", transform=None):
        self.dataset = OxfordIIITPet(
            root=root, target_types="segmentation", split=split, download=False
        )
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, mask = self.dataset[idx]
        image = np.array(image)
        mask = np.array(mask).astype(np.int64)
        
        # Oxford trimap fix
        mask = mask - 1

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented["mask"].long() # Ensure LongTensor for CE Loss

        return image, mask
    
# Use Nearest Neighbor for masks to keep class labels sharp
train_transform = A.Compose([
    A.Resize(128, 128),
    A.HorizontalFlip(p=0.5),
    A.Affine(
    translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)}, # shift_limit
    scale=(0.95, 1.05),                                       # scale_limit (1.0 +/- 0.05)
    rotate=(-15, 15),                                         # rotate_limit
    p=0.5
),
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])


# --- 2. Splitting Logic ---
# First, get the length of the dataset to create indices
base_dataset = OxfordIIITPet(root="/leonardo/home/userinternal/msafari1", split="trainval", target_types="category", download=False)
full_dataset_len = len(base_dataset)

#indices = torch.randperm(full_dataset_len)
#train_size = int(0.8 * len(base_dataset))
#train_idx = indices[:train_size]
#val_idx   = indices[train_size:]


#labels = base_dataset._labels
labels = [base_dataset[i][1] for i in range(len(base_dataset))]
indices = np.arange(full_dataset_len)
# Use train_test_split to get lists of integers
train_idx, val_idx = train_test_split(indices, train_size=0.8, random_state=42, stratify=np.array(labels))

# --- 3. Instantiate Datasets ---
# Define the actual dataset objects AFTER the split logic
train_dataset = AlbumentationsPetDataset(root="/leonardo/home/userinternal/msafari1", split="trainval", transform=train_transform)
val_dataset   = AlbumentationsPetDataset(root="/leonardo/home/userinternal/msafari1", split="trainval", transform=val_transform)

# --- 4. Create Subsets ---
# Use the indices integers directly here
train_subset = torch.utils.data.Subset(train_dataset, train_idx)
val_subset   = torch.utils.data.Subset(val_dataset, val_idx)

# Dedicated test set
test_dataset = AlbumentationsPetDataset(root="/leonardo/home/userinternal/msafari1", split="test", transform=val_transform)

# Settings
img_size   = 128
batch_size = 64
pin_memory = get_device() == "cuda"
num_workers = 0

# 5. Create DataLoaders

pet_train_loader = DataLoader(
    train_subset, batch_size=batch_size,
    shuffle=True, drop_last=True, 
    pin_memory=pin_memory, num_workers=num_workers
)

pet_val_loader = DataLoader(
    val_subset, batch_size=batch_size,
    shuffle=False, drop_last=False, 
    pin_memory=pin_memory, num_workers=num_workers
)

pet_test_loader = DataLoader(
    test_dataset, batch_size=batch_size,
    shuffle=False, drop_last=False, 
    pin_memory=pin_memory, num_workers=num_workers
)

img, msk = next(iter(pet_train_loader))
print(f"Image shape: {img.shape}") # Should be [64, 3, 128, 128]
print(f"Mask shape: {msk.shape}")   # Should be [64, 128, 128]
print(f"Mask unique values: {torch.unique(msk)}") # Should be [0, 1, 2]


Image shape: torch.Size([64, 3, 128, 128])
Mask shape: torch.Size([64, 128, 128])
Mask unique values: tensor([0, 1, 2])


In [14]:
# Settings
#img_size   = 128
#batch_size = 64
#pin_memory = True if get_device() == "cuda" else False
#num_workers = 0
#
# Dataset initialization
#raw_dataset = OxfordIIITPet(root=data_dir, download=True, target_types="segmentation")
#train_size = int(0.8 * len(raw_dataset))
#val_size = int(0.1 * len(raw_dataset))
#test_size = len(raw_dataset) - train_size - val_size
#
#train_raw, val_raw, test_raw = random_split(raw_dataset, [train_size, val_size, test_size])
#
# DataLoader with pin_memory=True (speeds up transfer to GPU/MPS)
#pet_train_loader = DataLoader(
#    PetSegmentationDataset(train_raw, img_size), batch_size=batch_size,
#    shuffle=True, drop_last=True, 
#    pin_memory=pin_memory, num_workers=num_workers
#)
#
#pet_val_loader = DataLoader(
#    PetSegmentationDataset(val_raw, img_size), batch_size=batch_size,
#    shuffle=False, drop_last=False, 
#    pin_memory=pin_memory, num_workers=num_workers
#)
#
#pet_test_loader = DataLoader(
#    PetSegmentationDataset(test_raw, img_size), batch_size=batch_size,
#    shuffle=False, drop_last=False, 
#    pin_memory=pin_memory, num_workers=num_workers
#)


In [15]:
def get_AdamW_CosineAnnealingLR(model, lr, weight_decay, T_max, eta_min):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_max, eta_min=eta_min)
    return optimizer, scheduler

def get_AdamW_ReduceLROnPlateau(model, lr, weight_decay, factor, patience, min_lr):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,  mode="min", factor=factor, patience=patience, min_lr=min_lr)
    return optimizer, scheduler

def get_AdamW_CosineAnnealingWarmRestarts(model, lr, weight_decay, T_0, T_mult, eta_min):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T_0, T_mult=T_mult, eta_min=eta_min)
    return optimizer, scheduler



In [ ]:
MAX_EPOCHS = 100

hparams_ignore = {
    "model": {"n_channels": 3, "n_classes": 3, "base_channel_size": 32},
    "training": {"max_epochs": MAX_EPOCHS, "device": str(device),}, 
#    "opt": {"lr": 1e-3, "weight_decay": 1e-2, "factor": 0.5, "patience":5, "min_lr":1e-6},
#    "opt": {"lr": 1e-3, "weight_decay": 1e-4, "T_max": MAX_EPOCHS, "eta_min":1e-6},
    "opt": {"lr": 1e-3, "weight_decay": 1e-2, "T_0": 8, "T_mult":2, "eta_min":1e-6},
    "loss": {"ce_weight": 0.5, "mode": "ignore_border"},
    "early_stopping":{"patience": 20, "min_delta": 0.001},
    "dataloader": {"img_size":img_size, "batch_size": batch_size, "pin_memory": pin_memory,
                   "num_workers": num_workers}
}

with open(dirs['unet_ignore_border']['configs'] / "hyperparams.json", "w") as f:
    json.dump(hparams_ignore, f)

    
unet_ignore = PetUNet(**hparams_ignore["model"])
print(torchinfo.summary(unet_ignore, input_size=(batch_size, 3, img_size, img_size)))
unet_ignore.to(device)

criterion_ignore = CombinedLoss(**hparams_ignore["loss"])
optimizer_ignore, scheduler_ignore = get_AdamW_CosineAnnealingWarmRestarts(unet_ignore, **hparams_ignore["opt"])
early_stopping_ignore = EarlyStopping(**hparams_ignore["early_stopping"])

writer_ignore = SummaryWriter(dirs['unet_ignore_border']['logs'])
checkpoint_filename_ignore = dirs['unet_ignore_border']['checkpoints'] / "model.pt"
result_filename_ignore = dirs['unet_ignore_border']['results'] / "result.json"

result_ignore  = train(pet_train_loader, pet_val_loader, pet_test_loader,
                unet_ignore, criterion_ignore,
                optimizer_ignore, scheduler_ignore, 
                MAX_EPOCHS, early_stopping_ignore,
                writer_ignore, checkpoint_filename_ignore, result_filename_ignore 
              )


Layer (type:depth-idx)                   Output Shape              Param #
PetUNet                                  [64, 3, 128, 128]         --
├─PetUNet_Encoder: 1-1                   [64, 512, 8, 8]           --
│    └─Sequential: 2-1                   [64, 32, 128, 128]        --
│    │    └─Conv2d: 3-1                  [64, 32, 128, 128]        864
│    │    └─BatchNorm2d: 3-2             [64, 32, 128, 128]        64
│    │    └─GELU: 3-3                    [64, 32, 128, 128]        --
│    │    └─Conv2d: 3-4                  [64, 32, 128, 128]        9,216
│    │    └─BatchNorm2d: 3-5             [64, 32, 128, 128]        64
│    │    └─GELU: 3-6                    [64, 32, 128, 128]        --
│    └─Sequential: 2-2                   [64, 64, 64, 64]          --
│    │    └─MaxPool2d: 3-7               [64, 32, 64, 64]          --
│    │    └─Conv2d: 3-8                  [64, 64, 64, 64]          18,432
│    │    └─BatchNorm2d: 3-9             [64, 64, 64, 64]          128
│    │

Epoch 000 [Train]:   0%|          | 0/46 [00:00<?, ?it/s]

In [ ]:
hparams_weighted = {
    "model": {"n_channels": 3, "n_classes": 3, "base_channel_size": 32},
    "training": {"max_epochs": MAX_EPOCHS, "device": str(device),}, 
#    "opt": {"lr": 1e-3, "weight_decay": 1e-2, "factor": 0.5, "patience":5, "min_lr":1e-6},
#    "opt": {"lr": 1e-3, "weight_decay": 1e-4, "T_max": MAX_EPOCHS, "eta_min":1e-6},
    "opt": {"lr": 1e-3, "weight_decay": 1e-2, "T_0": 8, "T_mult":2, "eta_min":1e-6},
    "loss": {"ce_weight": 0.5, "mode": "weighted_border"},
    "early_stopping":{"patience": 20, "min_delta": 0.001},
    "dataloader": {"img_size":img_size, "batch_size": batch_size, "pin_memory": pin_memory,
                   "num_workers": num_workers}
}

with open(dirs['unet_weighted_border']['configs'] / "hyperparams.json", "w") as f:
    json.dump(hparams_weighted, f)

    
unet_weighted = PetUNet(**hparams_weighted["model"])
print(torchinfo.summary(unet_weighted, input_size=(batch_size, 3, img_size, img_size)))
unet_weighted.to(device)

criterion_weighted = CombinedLoss(**hparams_weighted["loss"])
optimizer_weighted, scheduler_weighted = get_AdamW_CosineAnnealingWarmRestarts(unet_weighted, **hparams_weighted["opt"])
early_stopping_weighted = EarlyStopping(**hparams_weighted["early_stopping"])

writer_weighted = SummaryWriter(dirs['unet_weighted_border']['logs'])
checkpoint_filename_weighted = dirs['unet_weighted_border']['checkpoints'] / "model.pt"
result_filename_weighted = dirs['unet_weighted_border']['results'] / "result.json"

result_weighted  = train(pet_train_loader, pet_val_loader, pet_test_loader,
                unet_weighted, criterion_weighted,
                optimizer_weighted, scheduler_weighted, 
                MAX_EPOCHS, early_stopping_weighted,
                writer_weighted, checkpoint_filename_weighted, result_filename_weighted
              )


In [ ]:
hparams_balanced = {
    "model": {"n_channels": 3, "n_classes": 3, "base_channel_size": 32},
    "training": {"max_epochs": MAX_EPOCHS, "device": str(device),}, 
#    "opt": {"lr": 1e-3, "weight_decay": 1e-2, "factor": 0.5, "patience":5, "min_lr":1e-6},
#    "opt": {"lr": 1e-3, "weight_decay": 1e-4, "T_max": MAX_EPOCHS, "eta_min":1e-6},
    "opt": {"lr": 1e-3, "weight_decay": 1e-2, "T_0": 8, "T_mult":2, "eta_min":1e-6},
    "loss": {"ce_weight": 0.5, "mode": "balanced"},
    "early_stopping":{"patience": 25, "min_delta": 0.001},
    "dataloader": {"img_size":img_size, "batch_size": batch_size, "pin_memory": pin_memory,
                   "num_workers": num_workers}
}

with open(dirs['unet_balanced']['configs'] / "hyperparams.json", "w") as f:
    json.dump(hparams_balanced, f)

    
unet_balanced = PetUNet(**hparams_balanced["model"])
print(torchinfo.summary(unet_balanced, input_size=(batch_size, 3, img_size, img_size)))
unet_balanced.to(device)

criterion_balanced = CombinedLoss(**hparams_balanced["loss"])
optimizer_balanced, scheduler_balanced = get_AdamW_CosineAnnealingWarmRestarts(unet_balanced, **hparams_balanced["opt"])
early_stopping_balanced = EarlyStopping(**hparams_balanced["early_stopping"])

writer_balanced = SummaryWriter(dirs['unet_balanced']['logs'])
checkpoint_filename_balanced = dirs['unet_balanced']['checkpoints'] / "model.pt"
result_filename_balanced = dirs['unet_balanced']['results'] / "result.json"

result_balanced  = train(pet_train_loader, pet_val_loader, pet_test_loader,
                unet_balanced, criterion_balanced,
                optimizer_balanced, scheduler_balanced, 
                MAX_EPOCHS, early_stopping_balanced,
                writer_balanced, checkpoint_filename_balanced, result_filename_balanced 
              )


In [ ]:
def denormalize(img):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1).to(img.device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1).to(img.device)
    return img * std + mean

def visualize_results(test_loader, model, num_samples=5):
    model.eval()
    device = next(model.parameters()).device
    images, masks = next(iter(test_loader))
    images, masks = images.to(device), masks.to(device)

    with torch.no_grad():
        logits = model(images)
        preds = torch.argmax(logits, dim=1)

    # Move to CPU for plotting
    images = denormalize(images)
    images = images.clamp(0, 1)
    
    images = images.cpu().permute(0, 2, 3, 1).numpy()
    masks = masks.cpu().numpy()
    preds = preds.cpu().numpy()

    plt.figure(figsize=(15, num_samples * 4))
    
    for i in range(num_samples):
        # Original Image
        plt.subplot(num_samples, 3, i*3 + 1)
        plt.imshow(images[i])
        plt.title("Original Image")
        plt.axis("off")

        # Ground Truth Mask
        plt.subplot(num_samples, 3, i*3 + 2)
        plt.imshow(masks[i], cmap="viridis")
        plt.title("Ground Truth")
        plt.axis("off")

        # Model Prediction
        plt.subplot(num_samples, 3, i*3 + 3)
        plt.imshow(preds[i], cmap="viridis")
        plt.title("Predicted Mask")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_results(pet_test_loader, unet_balanced)

In [ ]:
visualize_results(pet_test_loader, unet_weighted)

In [ ]:
visualize_results(pet_test_loader, unet_ignore)

In [ ]:
def plot_confusion_matrix(test_loader, model):
    model.eval()
    device = next(model.parameters()).device
    all_preds = []
    all_masks = []
    
    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.append(preds.cpu().numpy().flatten())
            all_masks.append(masks.cpu().numpy().flatten())

    cm = confusion_matrix(
        np.concatenate(all_masks), 
        np.concatenate(all_preds), 
        labels=[0, 1, 2]
    )
    
    cm_perc = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    print(cm_perc)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_perc, annot=True, fmt='.2f', cmap="Blues",
                xticklabels=['Pet', 'BG', 'Border'], 
                yticklabels=['Pet', 'BG', 'Border'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()
    

In [ ]:
plot_confusion_matrix(pet_test_loader, unet_ignore)

In [ ]:
plot_confusion_matrix(pet_test_loader, unet_weighted)

In [ ]:
plot_confusion_matrix(pet_test_loader, unet_balanced)

# Evaluation Metrics for Semantic Segmentation: IoU and Dice Score

This code evaluates the performance of a semantic segmentation model by comparing the predicted masks with the ground-truth masks.

Semantic segmentation assigns a class label to every pixel in the image. 

Therefore, evaluation must be performed at the pixel level.


# 1. Intersection over Union (IoU)

## Definition

IoU, also called the **Jaccard Index**, measures the overlap between:

- predicted pixels belonging to a class
- true pixels belonging to the same class

It is defined as:

$
IoU = \frac{|Prediction \cap GroundTruth|}{|Prediction \cup GroundTruth|}
$

Where:

- **Intersection** = correctly predicted pixels of that class
- **Union** = pixels predicted as class OR truly belonging to class

## Interpretation

- IoU = 1 → perfect segmentation
- IoU = 0 → no overlap

Higher is better.

---

# 2. Dice Score

Dice Score (also called Sørensen–Dice coefficient or F1-score for segmentation) is:

$
Dice = \frac{2|Prediction \cap GroundTruth|}{|Prediction| + |GroundTruth|}
$

It gives more weight to overlap than IoU.

## Interpretation

- Dice = 1 → perfect overlap
- Dice = 0 → no overlap



In [ ]:
def compute_iou(pred, target, num_classes):
    ious = []

    pred = pred.view(-1)
    target = target.view(-1)

    for cls in range(num_classes):
        pred_i = pred == cls
        target_i = target == cls

        intersection = (pred_i & target_i).sum().item()
        union = (pred_i | target_i).sum().item()

        if union == 0:
            ious.append(float('nan'))  # class not present
        else:
            ious.append(intersection / union)

    return ious


def compute_dice(pred, target, num_classes):
    dices = []

    pred = pred.view(-1)
    target = target.view(-1)

    for cls in range(num_classes):
        pred_i = pred == cls
        target_i = target == cls

        intersection = (pred_i & target_i).sum().item()
        total = pred_i.sum().item() + target_i.sum().item()

        if total == 0:
            dices.append(float('nan'))
        else:
            dices.append(2 * intersection / total)

    return dices

`evaluate()` function evaluates a trained semantic segmentation model on a validation or test dataset using two common metrics:

- **IoU (Intersection over Union)**
- **Dice Score**

It computes these metrics **per class** and then averages them over the dataset.


In [ ]:
import torch
import numpy as np

def evaluate(model, dataloader, num_classes=3, mode="multiclass"):
    """
    mode:
        - "binary": assumes class 1 is foreground
        - "multiclass": full per-class evaluation
    """

    model.eval()
    device = next(model.parameters()).device

    total_inter = torch.zeros(num_classes, device=device)
    total_union = torch.zeros(num_classes, device=device)
    total_total = torch.zeros(num_classes, device=device)

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device).long()

            if masks.dim() == 4:
                masks = masks.squeeze(1)

            preds = torch.argmax(model(images), dim=1).long()

            for cls in range(num_classes):
                pred_i = (preds == cls)
                target_i = (masks == cls)

                inter = (pred_i & target_i).sum()
                union = (pred_i | target_i).sum()

                total_inter[cls] += inter
                total_union[cls] += union
                total_total[cls] += pred_i.sum() + target_i.sum()

    inter = total_inter.cpu().numpy()
    union = total_union.cpu().numpy()
    total = total_total.cpu().numpy()

    # --- metrics ---
    ious = np.divide(inter, union, out=np.zeros_like(inter, dtype=float), where=union != 0)
    dices = np.divide(2 * inter, total, out=np.zeros_like(inter, dtype=float), where=total != 0)

    # --- reporting ---
    classes = [f"Class {i}" for i in range(num_classes)]

    print("\n=== Metrics ===")

    if mode == "binary":
        # assume class 1 is foreground
        fg = 1 if num_classes > 1 else 0

        print(f"Foreground IoU  : {ious[fg]:.4f}")
        print(f"Foreground Dice : {dices[fg]:.4f}")

        # optional background report
        if num_classes > 1:
            print(f"Background IoU  : {ious[0]:.4f}")
            print(f"Background Dice : {dices[0]:.4f}")

        print(f"\nMean IoU (all classes): {ious[:2].mean():.4f}")
        print(f"Mean Dice (all classes): {dices[:2].mean():.4f}")

    else:
        for i in range(num_classes):
            print(f"{classes[i]} | IoU: {ious[i]:.4f} | Dice: {dices[i]:.4f}")

        print(f"\nMean IoU: {ious.mean():.4f}")
        print(f"Mean Dice: {dices.mean():.4f}")

    return ious, dices

In [ ]:
evaluate(unet_ignore, pet_test_loader, num_classes=3, mode="binary")


In [ ]:
evaluate(unet_weighted, pet_test_loader, num_classes=3, mode="multiclass")


In [ ]:
evaluate(unet_balanced, pet_test_loader, num_classes=3, mode="multiclass")


# Martino in Kyoto

<a title="Basile Morin, CC BY-SA 4.0 &lt;https://creativecommons.org/licenses/by-sa/4.0&gt;, via Wikimedia Commons" href="https://commons.wikimedia.org/wiki/File:Torii_path_with_lantern_at_Fushimi_Inari_Taisha_Shrine,_Kyoto,_Japan.jpg"><img  alt="Torii path with lantern at Fushimi Inari Taisha Shrine, Kyoto, Japan" src="https://upload.wikimedia.org/wikipedia/commons/thumb/0/0e/Torii_path_with_lantern_at_Fushimi_Inari_Taisha_Shrine%2C_Kyoto%2C_Japan.jpg/960px-Torii_path_with_lantern_at_Fushimi_Inari_Taisha_Shrine%2C_Kyoto%2C_Japan.jpg" width="400"></a>

In [ ]:
# ------------------------------------------
# 1. LOAD YOUR TRAINED MODEL (.pt)
# ------------------------------------------
with open(dirs['unet_balanced']['configs'] / "hyperparams.json", "r") as f:
    unet_config = json.load(f)
    
martino_unet = PetUNet(**unet_config["model"])
# 2. Load weights
state_dict = torch.load(checkpoint_filename_balanced, map_location=device)
martino_unet.load_state_dict(state_dict["model_state"])
martino_unet.to(device)



In [ ]:
# ------------------------------------------
# 2. LOAD IMAGES
# ------------------------------------------
# Load and prepare background
img_bg = Image.open("Torii_path_with_lantern_at_Fushimi_Inari_Taisha_Shrine_Kyoto_Japan.jpg").convert("RGB")
bg_np = np.array(img_bg)

# Load and rotate pet
img_pet = Image.open("martino.jpg").convert("RGB")
img_pet = img_pet.transpose(Image.ROTATE_270)
pet_np = np.array(img_pet)


In [ ]:
fig, axs = plt.subplots(1,2, figsize=(10,5))

for i, (img, title) in enumerate(zip([img_pet, img_bg], ["Martino", "Kyoto"])):
    axs[i].imshow(img)
    axs[i].set_title(title)
    axs[i].axis("off")
plt.show()


In [ ]:
def preprocess(img, model_img_size=128):
    transform = transforms.Compose([
        transforms.Resize((model_img_size, model_img_size)),
        transforms.ToTensor()
    ])

    img_tensor = transform(img).unsqueeze(0)
    return img_tensor


def predict(model, input_tensor):
    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        logits = model(input_tensor.to(device))
        preds = torch.argmax(logits, dim=1)
    return logits, preds


def show_images(images, titles):
    fig, axs = plt.subplots(1, len(images), figsize=(5 * len(images), 5))

    if len(images) == 1:
        axs = [axs]

    for i, (img, title) in enumerate(zip(images, titles)):
        axs[i].imshow(img)
        axs[i].set_title(title)
        axs[i].axis("off")

    plt.show()
    
    

In [ ]:
tensor_pet = preprocess(img_pet, 128)
logits, preds = predict(martino_unet, tensor_pet)
original_img = tensor_pet[0].permute(1, 2, 0).detach().cpu()
pred_mask = preds[0].detach().cpu()
show_images(
    [original_img, pred_mask],
    ["Original", "Segmentation"]
)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 4))
probs =  F.softmax(logits, dim=1).squeeze().detach().cpu()
for i, title in enumerate(["Pet", "BG", "Border"]):
    axs[i].imshow(probs[i], cmap="gray")
    axs[i].set_title(title)
    axs[i].axis("off")

plt.show()

In [ ]:
mask = (preds != 1).squeeze().detach().cpu().float()
mask = (preds == 0).squeeze().detach().cpu().float()
mask = (probs[0] > 0.1).float()

plt.imshow(mask, cmap="gray")
plt.title("Predicted Pet Mask")
plt.axis("off")
plt.show()


In [ ]:
def upscale_mask(img, mask):
    orig_w, orig_h = img.size

    mask_upscaled = F.interpolate(
        mask.unsqueeze(0).unsqueeze(0),
        size=(orig_h, orig_w),
        mode="bilinear",
        align_corners=False
    )[0, 0].cpu().numpy()

    return mask_upscaled

mask_upscaled = upscale_mask(img_pet, mask)
print(f"Mask shape before upscaling: {mask.shape}")
print(f"Mask shape after upscaling: {mask_upscaled.shape}")
plt.imshow(mask_upscaled, cmap="gray")
plt.title("Upscaled Mask")
plt.axis("off")
plt.show()


In [ ]:
def extract_scale(bg_np, pet_np, mask_upscaled, x, y):
    mask_3d = np.repeat(mask_upscaled[..., None], 3, axis=-1)

    pet_cut = (pet_np * mask_3d).astype(np.uint8)

    ph, pw = pet_cut.shape[:2]
    H, W = bg_np.shape[:2]

    x = min(max(x, 0), W - 10)
    y = min(max(y, 0), H - 10)

    scale = min((W - x) / pw, (H - y) / ph, 1.0)

    new_w = max(1, int(pw * scale))
    new_h = max(1, int(ph * scale))

    pet_cut_resized = np.array(
        Image.fromarray(pet_cut).resize((new_w, new_h), Image.BILINEAR)
    )

    mask_resized = np.array(
        Image.fromarray((mask_upscaled * 255).astype(np.uint8))
        .resize((new_w, new_h), Image.BILINEAR)
    ).astype(np.float32) / 255.0

    mask_resized_3d = np.repeat(mask_resized[..., None], 3, axis=-1)

    return pet_cut_resized, mask_resized_3d


In [ ]:
def show_rgb_debug(img, title):
    fig, axs = plt.subplots(1, 3, figsize=(9, 3))

    for i in range(3):
        axs[i].imshow(img[..., i], cmap="gray")
        axs[i].set_title(f"{title} ch {i}")
        axs[i].axis("off")

    plt.show()
    

In [ ]:
pet_cut_resized, mask_resized_3d = extract_scale(bg_np, pet_np, mask_upscaled, x=350, y=400)
show_rgb_debug(pet_cut_resized, "Pet Cut Resized")

show_rgb_debug(mask_resized_3d, "Mask Resized")


In [ ]:
def blend(bg_np, pet_cut_resized, mask_resized_3d, x, y):
    new_bg_np = bg_np.copy()

    h, w = pet_cut_resized.shape[:2]

    roi = bg_np[y:y+h, x:x+w]

    h = min(h, roi.shape[0])
    w = min(w, roi.shape[1])

    roi = roi[:h, :w]
    pet_cut_resized = pet_cut_resized[:h, :w]
    mask_resized_3d = mask_resized_3d[:h, :w]

    blended = (
        roi.astype(np.float32) * (1 - mask_resized_3d) +
        pet_cut_resized.astype(np.float32) * mask_resized_3d
    )

    new_bg_np[y:y+h, x:x+w] = np.clip(blended, 0, 255).astype(np.uint8)

    return new_bg_np


In [ ]:
martino_kyoto = blend(bg_np, pet_cut_resized, mask_resized_3d, x=350, y=400)

plt.figure(figsize=(8, 8))
plt.imshow(martino_kyoto)
plt.title("Martino in Kyoto")
plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
final_image = Image.fromarray(martino_kyoto)
martino_kyoto_filename = dirs['unet_balanced']['results'] / "MartinoInKyoto.png"
final_image.save(martino_kyoto_filename)
